# Decorators in Python

A **decorator** is a function that takes another function, adds some extra behaviour to it
(before / after it runs), and gives back a new function — **without changing the original function's code**.

To understand decorators we need 5 building blocks first:

1. Functions are objects (they can be stored in variables)
2. Scope — local vs global variables
3. Functions defined inside other functions
4. Functions returning functions
5. Functions passed as arguments (callbacks / higher-order functions)

Then we combine all of them to build a decorator.

## 1. A simple function

`def` creates a function object and stores it under a name. Calling it with `()` runs its body.

In [1]:
def func():      # 'def' creates a function object and binds it to the name 'func'
    return 1     # 'return' sends the value 1 back to whoever called the function

In [2]:
func()           # the () CALLS the function -> runs the body and gives back 1

1

## 2. Scope — local vs global

* **Global variable** → created outside every function, visible everywhere in the file.
* **Local variable** → created inside a function, exists only while that function runs.
* `locals()` returns a dictionary of the variables in the current (local) scope.
* `globals()` returns a dictionary of all global names in the file/notebook.

Python looks up names in the order **L → E → G → B** (Local, Enclosing, Global, Built-in).

In [5]:
s = "Global variable"      # defined outside any function -> global variable

def check_for_locals():
    x = "im local"         # defined inside the function -> local variable
    print(locals())        # prints a dict of ONLY the local variables: {'x': 'im local'}

In [6]:
check_for_locals()         # running the function shows its local scope

{'x': 'im local'}


In [7]:
print(globals())           # dict of every global name (variables, functions, imports, notebook internals...)

{'__name__': '__main__', '__doc__': 'Automatically created module for IPython interactive environment', '__package__': None, '__loader__': None, '__spec__': None, '__builtin__': <module 'builtins' (built-in)>, '__builtins__': <module 'builtins' (built-in)>, '_ih': ['', 'def func():\n    return 1', 'func()', 's = "Global variable" # global variable\ndef check_for_locals():\n    print(locals())', 'check_for_locals()', 's = "Global variable" # global variable\ndef check_for_locals():\n    x = "im local"\n    print(locals())', 'check_for_locals()', 'print(globals())'], '_oh': {2: 1}, '_dh': [WindowsPath('C:/Users/rohn1/Desktop/classes/parminder py sessions')], 'In': ['', 'def func():\n    return 1', 'func()', 's = "Global variable" # global variable\ndef check_for_locals():\n    print(locals())', 'check_for_locals()', 's = "Global variable" # global variable\ndef check_for_locals():\n    x = "im local"\n    print(locals())', 'check_for_locals()', 'print(globals())'], 'Out': {2: 1}, 'get_ip

In [8]:
globals()["s"]             # globals() is a normal dict, so we can read our global 's' by key

'Global variable'

## 3. Functions are objects

A function name is just a **label pointing to a function object**.
We can assign that object to another name, and both names point to the same function.

In [11]:
def hello(name = "jose"):        # 'name' is a parameter with a default value "jose"
    return "hello " + name       # join the strings and return the result

In [10]:
hello()                          # no argument passed -> default "jose" is used

'hellojose'

In [12]:
greet = hello                    # NO () here -> we are NOT calling hello, we copy the reference to the function object

In [13]:
greet                            # shows the function object itself (note: its name is still 'hello')

<function __main__.hello(name='jose')>

In [14]:
greet()                          # calling through the new label works exactly like hello()

'hello jose'

In [16]:
del hello                        # deletes only the NAME 'hello', not the function object

In [17]:
hello()                          # NameError -> the label 'hello' no longer exists (this error is expected)

NameError: name 'hello' is not defined

In [18]:
greet()                          # still works! 'greet' still points to the function object

'hello jose'

## 4. Functions inside functions

We can define functions inside another function.
The inner functions are **local** to the outer function — they can't be used from outside.

In [23]:
def hello (name = "jose"):
    print("The hello() function has been executed")    # runs as soon as hello() is called

    def greeting():                                     # inner function -> exists only inside hello()
        return "\t this is inside the greet() function"

    def welcome():                                      # another inner function
        return "\t this is inside the welcome() function"

    print(greeting())                                   # inner functions can be called from inside hello()
    print(welcome())
    print("now we are back inside the hello() function")

In [24]:
hello()                          # runs hello(), which internally creates and calls both inner functions

The hello() function has been executed
	 this is inside the greet() function
	 this is inside the welcome() function
now we are back inside the hello() function


In [25]:
greeting()                       # NameError -> greeting is local to hello(), not visible outside (expected error)

NameError: name 'greeting' is not defined

In [26]:
welcome()                        # NameError for the same reason (expected error)

NameError: name 'welcome' is not defined

## 5. Returning functions

Instead of calling an inner function, the outer function can **return the inner function object**
(name without `()`). The caller receives a function and can call it later.

In [28]:
def hello (name = "jose"):
    print("The hello() function has been executed")

    def greeting():
        return "\t this is inside the greet() function"

    def welcome():
        return "\t this is inside the welcome() function"

    if name == "jose":           # choose which inner function to hand back
        return greeting          # no () -> return the function itself, not its result
    else:
        return welcome

In [29]:
res = hello()                    # hello() runs and returns the 'greeting' function -> stored in res

The hello() function has been executed


In [32]:
print(res())                     # res is now a function, so res() calls greeting()

	 this is inside the greet() function


In [33]:
res2 = hello("rudr")             # name != "jose" -> welcome function is returned
print(res2())                    # call the returned welcome function

The hello() function has been executed
	 this is inside the welcome() function


In [34]:
hello()()                        # first () calls hello -> returns greeting; second () calls greeting

The hello() function has been executed


'\t this is inside the greet() function'

In [35]:
hello("rudra")()                 # same idea: hello returns welcome, then welcome is called immediately

The hello() function has been executed


'\t this is inside the welcome() function'

## 6. Functions as arguments

* **Higher-Order Function (HOF):** a function that takes a function as an argument and/or returns a function.
* **Callback:** the function that is passed into another function, to be "called back" later.

In [37]:
# hof : a function that accepts another function as argument or returns a function (e.g. other, map, filter)
# callback : the function we pass into the HOF so it can be called later (e.g. hello below)

In [39]:
def hello ():
    return "Hi jose"

def other(cb):                               # 'cb' will receive a FUNCTION (callback)
    print("Other code would go here...")
    print(cb())                              # call the received function and print its result

In [40]:
other(hello)                     # pass hello WITHOUT () -> we pass the function, not its result

Other code would go here...
Hi jose


### Practice: callback example

In [41]:
# create a function called lifespan , which takes average_age, yob , and a function returning age of a person

In [43]:
from datetime import date                    # date class gives today's date

def ageCal(yob):                             # yob = year of birth
    return date.today().year - yob           # current year - birth year = age

In [44]:
def lifeSpan(avg_age, yob, cb):              # cb = any function that turns yob into age
    return avg_age - cb(yob)                 # years left = average age - current age

In [45]:
lifeSpan(90 , 1997 ,  ageCal)                # pass the named function ageCal as the callback

61

In [46]:
lifeSpan(90 , 1997 , lambda yob : date.today().year - yob)   # same thing using an anonymous (lambda) function

61

## 7. Creating a decorator

Now we combine everything:

1. The decorator **takes a function** as argument (functions as arguments)
2. It **defines a wrapper function inside** (nested functions)
3. The wrapper runs extra code **before and after** calling the original function
4. The decorator **returns the wrapper** (returning functions)
5. We re-assign the original name to the wrapper (functions are objects)

The inner `wrap_func` still remembers `func` even after `new_decorator` has finished — this is called a **closure**.

In [48]:
def new_decorator(func):                     # func = the original function we want to decorate

    def wrap_func():                         # wrapper: the new, "decorated" version of func
        print("code would be here, before executing the func")   # extra behaviour BEFORE
        func()                                                   # call the original function
        print("code here will execute after the func()")         # extra behaviour AFTER

    return wrap_func                         # return the wrapper (not called) to the caller

In [49]:
def func_needs_decorator():
    print("I NEED DECORATION")

In [52]:
# manual decoration: pass the function in, get the wrapper back, and store it under the SAME name
func_needs_decorator = new_decorator(func_needs_decorator)

In [53]:
func_needs_decorator()           # now calls wrap_func -> before text, original function, after text

code would be here, before executing the func
I NEED DECORATION
code here will execute after the func()


### Another decorator: draw a border around returned text

In [69]:
def please_decrate():
    return "Decorate me i need a lot of decoration"

In [70]:
def my_decor(cb):                            # cb = function that returns a string
    def decorated():
        print("*" * len(cb()))               # a line of * as long as the returned text
        print(cb())                          # the original text
        print("*" * len(cb()))               # bottom border

    return decorated                         # return the wrapper

In [71]:
my_decor(please_decrate)()                   # decorate and call in one line: my_decor(...) returns decorated, () calls it

**************************************
Decorate me i need a lot of decoration
**************************************


## 8. The `@` syntax (syntactic sugar)

Writing `@my_decor` above a function is exactly the same as writing
`i_need_decor = my_decor(i_need_decor)` right after defining it.

In [72]:
@my_decor                                    # same as: i_need_decor = my_decor(i_need_decor)
def i_need_decor():
    return "Please decorate me"

In [73]:
i_need_decor()                               # actually runs the 'decorated' wrapper

******************
Please decorate me
******************


## 9. Real-life use cases — Tasks (try them yourself)

Use only what we covered above: functions as objects, nested functions, returning functions,
callbacks, and decorators written with a wrapper (`wrap_func`) and the `@` syntax.

### Task 1 — Login required
Create a global variable `is_logged_in = False`.
Write a decorator `login_required` whose wrapper checks `is_logged_in` before calling the function:
if `True` → call the function, otherwise print `"Please log in first"`.
Use it with `@` on `view_cart()` and `checkout()` (both just print a message).
Call them, then change `is_logged_in` to `True` and call them again.

### Task 2 — Shop bill border
Write a function `bill()` that **returns** a string like `"Total amount: Rs 2450"`.
Write a decorator `bill_border` (like `my_decor`) whose wrapper prints:
the shop name on top, a line of `=` the same length as the bill text, the bill text,
another line of `=`, and `"Thank you for shopping!"` at the bottom.

### Task 3 — How long did it take?
Write a decorator `timer` whose wrapper records the time before and after calling the function
(`import time` and use `time.time()`), then prints `"Took X seconds"`.
Decorate a function `count_to_million()` that loops from 1 to 1,000,000 and prints `"Done counting"`.

### Task 4 — Discount picker (returning functions + callbacks)
Write a function `get_discount(customer_type)` that defines two inner functions,
`regular_discount(amount)` (5% off) and `premium_discount(amount)` (20% off),
and **returns** one of them based on `customer_type` (like `hello()` returning `greeting` / `welcome`).
Then write `final_bill(amount, cb)` that uses the callback `cb` to calculate and return the amount to pay
(like `lifeSpan`). Test it with `final_bill(1000, get_discount("premium"))`.
